# Real World Text Processing and Encoding

This notebook walks through the full pipeline step by step, starting with raw Amazon reviews and ending with feature engineering and sentiment classification.

What is covered here:
- loading the raw review file
- cleaning and preprocessing the text
- building a vocabulary
- creating BoW, TF-IDF, and one-hot features
- comparing the feature representations
- training a simple sentiment classifier

Files used in this project:
- input: `amazon_raw_reviews.csv`
- processed output: `amazon_processed_reviews.csv`

In [18]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd

# Point to the project folder and the files we care about in this first step.
project_dir = Path(r"D:\GEN AI Projects\Embedding_Encoding\Project-Real_World_TextProcssing & Encoding")
raw_file = project_dir / "amazon_raw_reviews.csv"
default_processed_file = project_dir / "amazon_processed_reviews.csv"

# Leave this as False if you do not want to overwrite an existing processed file.
overwrite_existing = False


# Basic text cleanup for the raw review fields.
# This removes the trailing "Read more", fixes a common encoding issue,
# and trims extra spaces.
def clean_text(value):
    text = "" if pd.isna(value) else str(value)
    text = text.replace("�", "'")
    text = re.sub(r"Read more\s*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Try the usual UTF-8 read first, then fall back to cp1252 if needed.
try:
    reviews_df = pd.read_csv(raw_file, encoding="utf-8")
except UnicodeDecodeError:
    reviews_df = pd.read_csv(raw_file, encoding="cp1252")

# Rename the columns so the rest of the notebook uses a consistent schema.
reviews_df = reviews_df.rename(
    columns={
        "Star Rating": "star_rating",
        "Review Title": "review_title",
        "Review Description": "review_description",
    }
)

# If any expected column is missing, create it so the pipeline does not break.
for column in ["star_rating", "review_title", "review_description"]:
    if column not in reviews_df.columns:
        reviews_df[column] = ""

# Clean the individual fields before combining them into one review text column.
reviews_df["review_title"] = reviews_df["review_title"].map(clean_text)
reviews_df["review_description"] = reviews_df["review_description"].map(clean_text)
reviews_df["star_rating"] = reviews_df["star_rating"].map(clean_text)

# Combine title and description into one text field for the later NLP steps.
reviews_df["review_text"] = (
    reviews_df["review_title"].fillna("") + " " + reviews_df["review_description"].fillna("")
).map(clean_text)

# Keep only the cleaned rows that still have usable review text.
processed_df = reviews_df.loc[
    reviews_df["review_text"].str.len() > 0,
    ["review_text", "review_title", "review_description", "star_rating"],
].copy()

# If a processed file is already present, create a timestamped copy instead
# unless overwrite_existing has been set to True.
if default_processed_file.exists() and not overwrite_existing:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    processed_file = project_dir / f"amazon_processed_reviews_{timestamp}.csv"
    print("A processed file already exists, so a timestamped version will be saved.")
else:
    processed_file = default_processed_file
    if processed_file.exists():
        print("A processed file already exists and will be overwritten.")
    else:
        print("No processed file was found, so a new one will be created.")

# Save the cleaned dataset into the project folder.
processed_df.to_csv(processed_file, index=False, encoding="utf-8")

# Quick confirmation that the file was created successfully.
print(f"Processed file created: {processed_file}")
print(f"File exists: {processed_file.exists()}")
print(f"Total cleaned reviews: {len(processed_df)}")

A processed file already exists, so a timestamped version will be saved.
Processed file created: D:\GEN AI Projects\Embedding_Encoding\Project-Real_World_TextProcssing & Encoding\amazon_processed_reviews_20260410_140249.csv
File exists: True
Total cleaned reviews: 696


In [19]:
# First, make everything lowercase so words like 'Good' and 'good'
# are treated as the same token.
step1_lowercase_df = processed_df.copy()
step1_lowercase_df["lowercase_text"] = step1_lowercase_df["review_text"].astype(str).str.lower()

step1_lowercase_df[["review_text", "lowercase_text"]].head(10)

,review_text,lowercase_text
0,"Great product, you should buy for it if you ar...","great product, you should buy for it if you ar..."
1,It is THE MIC that you need for everyday conte...,it is the mic that you need for everyday conte...
2,Good mic but Bad magnet Pros - It is a great p...,good mic but bad magnet pros - it is a great p...
3,Quality is not good Very Bad Quality product f...,quality is not good very bad quality product f...
4,Purchase it without any hesitation 1st class p...,purchase it without any hesitation 1st class p...
5,Value for money Value for money! Good voice qu...,value for money value for money! good voice qu...
6,"Quality Highly recommend this product ,it's re...","quality highly recommend this product ,it's re..."
7,Noice cancellation works only with settings Pl...,noice cancellation works only with settings pl...
8,Nice product. I specially came to amazon to fi...,nice product. i specially came to amazon to fi...
9,"Good product Very good item i used it , but i ...","good product very good item i used it , but i ..."


In [20]:
# Split each review into individual word tokens.
step2_tokenized_df = step1_lowercase_df.copy()
step2_tokenized_df["tokens"] = step2_tokenized_df["lowercase_text"].apply(lambda text: text.split())

step2_tokenized_df[["lowercase_text", "tokens"]].head(5)

,lowercase_text,tokens
0,"great product, you should buy for it if you ar...","[great, product,, you, should, buy, for, it, i..."
1,it is the mic that you need for everyday conte...,"[it, is, the, mic, that, you, need, for, every..."
2,good mic but bad magnet pros - it is a great p...,"[good, mic, but, bad, magnet, pros, -, it, is,..."
3,quality is not good very bad quality product f...,"[quality, is, not, good, very, bad, quality, p..."
4,purchase it without any hesitation 1st class p...,"[purchase, it, without, any, hesitation, 1st, ..."


In [21]:
# Remove punctuation from each token and drop anything that becomes empty after cleanup.
step3_no_punct_df = step2_tokenized_df.copy()
step3_no_punct_df["tokens_no_punct"] = step3_no_punct_df["tokens"].apply(
    lambda tokens: [re.sub(r"[^a-z0-9]", "", token) for token in tokens]
)
step3_no_punct_df["tokens_no_punct"] = step3_no_punct_df["tokens_no_punct"].apply(
    lambda tokens: [token for token in tokens if token]
)

step3_no_punct_df[["tokens", "tokens_no_punct"]].head(5)

,tokens,tokens_no_punct
0,"[great, product,, you, should, buy, for, it, i...","[great, product, you, should, buy, for, it, if..."
1,"[it, is, the, mic, that, you, need, for, every...","[it, is, the, mic, that, you, need, for, every..."
2,"[good, mic, but, bad, magnet, pros, -, it, is,...","[good, mic, but, bad, magnet, pros, it, is, a,..."
3,"[quality, is, not, good, very, bad, quality, p...","[quality, is, not, good, very, bad, quality, p..."
4,"[purchase, it, without, any, hesitation, 1st, ...","[purchase, it, without, any, hesitation, 1st, ..."


In [22]:
# Drop common stopwords so the remaining tokens focus more on useful content words.
stop_words = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from",
    "has", "have", "in", "is", "it", "its", "of", "on", "or", "that", "the",
    "to", "was", "were", "will", "with"
}

step4_no_stopwords_df = step3_no_punct_df.copy()
step4_no_stopwords_df["tokens_without_stopwords"] = step4_no_stopwords_df["tokens_no_punct"].apply(
    lambda tokens: [token for token in tokens if token not in stop_words]
)

step4_no_stopwords_df[["tokens_no_punct", "tokens_without_stopwords"]].head(5)

,tokens_no_punct,tokens_without_stopwords
0,"[great, product, you, should, buy, for, it, if...","[great, product, you, should, buy, if, you, be..."
1,"[it, is, the, mic, that, you, need, for, every...","[mic, you, need, everyday, content, shooting, ..."
2,"[good, mic, but, bad, magnet, pros, it, is, a,...","[good, mic, bad, magnet, pros, great, product,..."
3,"[quality, is, not, good, very, bad, quality, p...","[quality, not, good, very, bad, quality, produ..."
4,"[purchase, it, without, any, hesitation, 1st, ...","[purchase, without, any, hesitation, 1st, clas..."


## Vocabulary Creation

Now that the text has been cleaned, the next step is to look at the words that actually make up the dataset.

In this part, the notebook:
- builds the vocabulary from the cleaned tokens
- prints the total vocabulary size
- shows a small sample of the vocabulary
- highlights the most frequent words in the corpus

In [23]:
# Build the vocabulary from the cleaned tokens.
# The vocabulary is just the full set of unique words seen in the corpus.
all_tokens = step4_no_stopwords_df["tokens_without_stopwords"].sum()
vocabulary = sorted(set(all_tokens))

print(f"Vocabulary size: {len(vocabulary)}")
print("Sample vocabulary words:")
print(vocabulary[:30])

Vocabulary size: 4750
Sample vocabulary words:
['0', '0000001', '010', '051', '1', '10', '100', '100transparency', '1010', '1010highly', '1010sound', '1015', '101v2', '1040', '10k', '10k12k', '10mm', '10x', '11000', '1110', '1110bass', '111v2', '1120', '11mm', '12', '120', '120hz', '1212', '124mm', '128']


In [7]:
# Count how often each word shows up after preprocessing.
from collections import Counter

word_frequencies = Counter(all_tokens)
top_frequent_words_df = pd.DataFrame(
    word_frequencies.most_common(20),
    columns=["word", "frequency"]
)

top_frequent_words_df

,word,frequency
0,i,859
1,good,848
2,quality,737
3,sound,647
4,this,544
5,earbuds,543
6,product,453
7,very,422
8,battery,415
9,not,364


## Feature Engineering

After preprocessing, the text needs to be converted into numbers so a model can work with it.

This part creates three different representations:
- one-hot encoding
- bag of words
- TF-IDF

All three are built from the same cleaned text so the comparison stays fair.

In [24]:
# Turn the cleaned token lists back into strings so sklearn vectorizers can use them.
feature_df = step4_no_stopwords_df.copy()
feature_df["final_preprocessed_text"] = feature_df["tokens_without_stopwords"].apply(lambda tokens: " ".join(tokens))

feature_df[["review_text", "final_preprocessed_text"]].head(5)

,review_text,final_preprocessed_text
0,"Great product, you should buy for it if you ar...",great product you should buy if you beginner g...
1,It is THE MIC that you need for everyday conte...,mic you need everyday content shooting video c...
2,Good mic but Bad magnet Pros - It is a great p...,good mic bad magnet pros great product this pr...
3,Quality is not good Very Bad Quality product f...,quality not good very bad quality product holl...
4,Purchase it without any hesitation 1st class p...,purchase without any hesitation 1st class prod...


In [25]:
# One-hot encoding marks whether a word appears in a review or not.
try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
except ImportError as exc:
    raise ImportError("Install scikit-learn in the notebook environment: pip install scikit-learn") from exc

one_hot_vectorizer = CountVectorizer(binary=True)
one_hot_matrix = one_hot_vectorizer.fit_transform(feature_df["final_preprocessed_text"])
one_hot_feature_names = one_hot_vectorizer.get_feature_names_out()

print(f"One Hot matrix shape: {one_hot_matrix.shape}")

one_hot_preview_df = pd.DataFrame(
    one_hot_matrix.toarray()[:5],
    columns=one_hot_feature_names
)
one_hot_preview_df.iloc[:, :15]

One Hot matrix shape: (696, 4724)


,0000001,010,051,10,100,100transparency,1010,1010highly,1010sound,1015,101v2,1040,10k,10k12k,10mm
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
# Bag of Words keeps track of how many times each word appears in each review.
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(feature_df["final_preprocessed_text"])
bow_feature_names = bow_vectorizer.get_feature_names_out()

print(f"Bag of Words matrix shape: {bow_matrix.shape}")

bow_preview_df = pd.DataFrame(
    bow_matrix.toarray()[:5],
    columns=bow_feature_names
)
bow_preview_df.iloc[:, :15]

Bag of Words matrix shape: (696, 4724)


,0000001,010,051,10,100,100transparency,1010,1010highly,1010sound,1015,101v2,1040,10k,10k12k,10mm
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [28]:
# TF-IDF gives higher weight to words that are important in a review
# and lower weight to words that appear everywhere.
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(feature_df["final_preprocessed_text"])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

tfidf_preview_df = pd.DataFrame(
    tfidf_matrix.toarray()[:5],
    columns=tfidf_feature_names
)
tfidf_preview_df.iloc[:, :15]

TF-IDF matrix shape: (696, 4724)


,0000001,010,051,10,100,100transparency,1010,1010highly,1010sound,1015,101v2,1040,10k,10k12k,10mm
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Comparison Analysis

Once the three feature sets are ready, it helps to compare them side by side.

This part of the notebook:
- summarizes how OHE, BoW, and TF-IDF differ
- looks at which words get the highest TF-IDF scores
- explains why common words usually receive lower TF-IDF weight

In [12]:
# Put the three feature methods side by side for an easy comparison.
comparison_df = pd.DataFrame([
    {
        "Method": "One Hot Encoding",
        "Value Type": "0 or 1",
        "Meaning": "Shows whether a word is present in a document",
        "Matrix Shape": str(one_hot_matrix.shape),
        "Sparsity": f"{(1 - one_hot_matrix.nnz / (one_hot_matrix.shape[0] * one_hot_matrix.shape[1])):.2%}",
    },
    {
        "Method": "Bag of Words",
        "Value Type": "Word count",
        "Meaning": "Counts how many times a word appears in a document",
        "Matrix Shape": str(bow_matrix.shape),
        "Sparsity": f"{(1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])):.2%}",
    },
    {
        "Method": "TF-IDF",
        "Value Type": "Weighted score",
        "Meaning": "Measures word importance using term frequency and inverse document frequency",
        "Matrix Shape": str(tfidf_matrix.shape),
        "Sparsity": f"{(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.2%}",
    },
])

comparison_df

,Method,Value Type,Meaning,Matrix Shape,Sparsity
0,One Hot Encoding,0 or 1,Shows whether a word is present in a document,"(696, 4724)",98.93%
1,Bag of Words,Word count,Counts how many times a word appears in a docu...,"(696, 4724)",98.93%
2,TF-IDF,Weighted score,Measures word importance using term frequency ...,"(696, 4724)",98.93%


In [29]:
# Look at the words with the highest average TF-IDF scores across the dataset.
average_tfidf_scores = tfidf_matrix.mean(axis=0).A1
average_tfidf_df = pd.DataFrame({
    "word": tfidf_feature_names,
    "average_tfidf_score": average_tfidf_scores,
}).sort_values("average_tfidf_score", ascending=False)

top_tfidf_words_df = average_tfidf_df.head(20).reset_index(drop=True)
print("Top words by average TF-IDF score:")
display(top_tfidf_words_df)

print(
    "TF-IDF gives more weight to words that matter in a smaller set of documents and less weight to "
    "words that show up almost everywhere. That is why very common words usually end up with lower scores."
)

Top words by average TF-IDF score:


,word,average_tfidf_score
0,good,0.083590
1,quality,0.058501
2,product,0.054875
3,sound,0.049638
4,very,0.042551
5,this,0.038919
6,battery,0.037913
7,earbuds,0.033594
8,great,0.032882
9,not,0.031500


TF-IDF gives more weight to words that matter in a smaller set of documents and less weight to words that show up almost everywhere. That is why very common words usually end up with lower scores.


## Sparse Matrix Analysis

Text vectors usually end up being very sparse, which means most of the values are zero.

This section checks:
- the shape of each matrix
- how sparse each representation is
- why high-dimensional sparse features can still become costly in large systems

In [31]:
# Check how sparse each matrix is.
# In text data, most values are zero because each review uses only a small part of the vocabulary.

def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero_elements = matrix.nnz
    zero_elements = total_elements - non_zero_elements
    return (zero_elements / total_elements) * 100

sparse_analysis_df = pd.DataFrame([
    {
        "Method": "One Hot Encoding",
        "Shape": str(one_hot_matrix.shape),
        "Non-Zero Values": int(one_hot_matrix.nnz),
        "Sparsity (%)": round(calculate_sparsity(one_hot_matrix), 2),
    },
    {
        "Method": "Bag of Words",
        "Shape": str(bow_matrix.shape),
        "Non-Zero Values": int(bow_matrix.nnz),
        "Sparsity (%)": round(calculate_sparsity(bow_matrix), 2),
    },
    {
        "Method": "TF-IDF",
        "Shape": str(tfidf_matrix.shape),
        "Non-Zero Values": int(tfidf_matrix.nnz),
        "Sparsity (%)": round(calculate_sparsity(tfidf_matrix), 2),
    },
])

print("Sparse matrix analysis:")
display(sparse_analysis_df)

print(
    "These matrices are efficient to store in sparse form, but they can still become expensive at large "
    "scale because the feature space grows quickly with vocabulary size. They become especially costly if "
    "you ever convert them into dense arrays."
)

Sparse matrix analysis:


,Method,Shape,Non-Zero Values,Sparsity (%)
0,One Hot Encoding,"(696, 4724)",35198,98.93
1,Bag of Words,"(696, 4724)",35198,98.93
2,TF-IDF,"(696, 4724)",35198,98.93


These matrices are efficient to store in sparse form, but they can still become expensive at large scale because the feature space grows quickly with vocabulary size. They become especially costly if you ever convert them into dense arrays.


## Sentiment Classification

The last modeling step is a simple positive-vs-negative sentiment classifier built with Logistic Regression.

To keep the labels clean:
- ratings of 4 or 5 are treated as positive
- ratings of 1 or 2 are treated as negative
- ratings of 3 are left out

The model is trained twice so the feature representations can be compared directly:
- Bag of Words
- TF-IDF

In [32]:
# Build simple binary sentiment labels from the star ratings.
# Ratings 4 and 5 become positive, ratings 1 and 2 become negative,
# and rating 3 is left out to avoid mixing neutral reviews into the task.
feature_df = feature_df.copy()
feature_df["numeric_rating"] = feature_df["star_rating"].str.extract(r"(\d+(?:\.\d+)?)").astype(float)

sentiment_mask = (feature_df["numeric_rating"] >= 4) | (feature_df["numeric_rating"] <= 2)
sentiment_df = feature_df.loc[sentiment_mask].copy()
sentiment_df["sentiment_label"] = sentiment_df["numeric_rating"].apply(
    lambda rating: 1 if rating >= 4 else 0
)

sentiment_df[["star_rating", "numeric_rating", "sentiment_label"]].head(10)

,star_rating,numeric_rating,sentiment_label
0,5.0 out of 5 stars,5.0,1
1,5.0 out of 5 stars,5.0,1
2,4.0 out of 5 stars,4.0,1
3,1.0 out of 5 stars,1.0,0
4,5.0 out of 5 stars,5.0,1
5,5.0 out of 5 stars,5.0,1
6,5.0 out of 5 stars,5.0,1
7,4.0 out of 5 stars,4.0,1
8,5.0 out of 5 stars,5.0,1
9,5.0 out of 5 stars,5.0,1


In [35]:
# Train Logistic Regression twice so the BoW and TF-IDF representations
# can be compared on the same train/test split.
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, classification_report
    from sklearn.model_selection import train_test_split
except ImportError as exc:
    raise ImportError("Install scikit-learn in the notebook environment: pip install scikit-learn") from exc

filtered_indices = sentiment_df.index.to_numpy()
y = sentiment_df["sentiment_label"].to_numpy()

bow_filtered = bow_matrix[filtered_indices]
tfidf_filtered = tfidf_matrix[filtered_indices]

train_indices, test_indices = train_test_split(
    range(len(y)),
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train_bow = bow_filtered[train_indices]
X_test_bow = bow_filtered[test_indices]
X_train_tfidf = tfidf_filtered[train_indices]
X_test_tfidf = tfidf_filtered[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

bow_model = LogisticRegression(max_iter=1000)
bow_model.fit(X_train_bow, y_train)
bow_predictions = bow_model.predict(X_test_bow)

tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(X_train_tfidf, y_train)
tfidf_predictions = tfidf_model.predict(X_test_tfidf)

comparison_results_df = pd.DataFrame([
    {
        "Feature Type": "Bag of Words",
        "Accuracy": round(accuracy_score(y_test, bow_predictions), 4),
    },
    {
        "Feature Type": "TF-IDF",
        "Accuracy": round(accuracy_score(y_test, tfidf_predictions), 4),
    },
])

print("Performance comparison:")
display(comparison_results_df)

print("Bag of Words classification report:")
print(classification_report(y_test, bow_predictions, target_names=["Negative", "Positive"]))

print("TF-IDF classification report:")
print(classification_report(y_test, tfidf_predictions, target_names=["Negative", "Positive"]))

Performance comparison:


,Feature Type,Accuracy
0,Bag of Words,0.9524
1,TF-IDF,0.9127


Bag of Words classification report:
              precision    recall  f1-score   support

    Negative       0.80      0.67      0.73        12
    Positive       0.97      0.98      0.97       114

    accuracy                           0.95       126
   macro avg       0.88      0.82      0.85       126
weighted avg       0.95      0.95      0.95       126

TF-IDF classification report:
              precision    recall  f1-score   support

    Negative       1.00      0.08      0.15        12
    Positive       0.91      1.00      0.95       114

    accuracy                           0.91       126
   macro avg       0.96      0.54      0.55       126
weighted avg       0.92      0.91      0.88       126



## Final Conclusion

This last section gives a short summary of which feature representation performed better in the sentiment experiment and what that result means in practical terms.

In [36]:
# Wrap up the experiment by identifying which feature representation gave the better accuracy.
best_model_row = comparison_results_df.sort_values("Accuracy", ascending=False).iloc[0]
best_feature_type = best_model_row["Feature Type"]
best_accuracy = best_model_row["Accuracy"]

print(f"Best performing feature representation: {best_feature_type}")
print(f"Best accuracy: {best_accuracy}")
print()
print(
    f"Final takeaway: {best_feature_type} gave the better result for this sentiment task. "
    "That usually means its representation preserved more useful signal for Logistic Regression on this dataset."
)

Best performing feature representation: Bag of Words
Best accuracy: 0.9524

Final takeaway: Bag of Words gave the better result for this sentiment task. That usually means its representation preserved more useful signal for Logistic Regression on this dataset.
